# Download IG price history

Pulls **OHLC candles** from the IG REST API — one market, one resolution,
over a time window you choose — then optionally writes them to a timestamped
CSV.

**Step 1 of a four-notebook pipeline:**

- **`01` download** (this notebook) — pull candles, write the CSV
- **`02` import CSV** — load the CSV into `data/ig_market_data.db`
- **`03` view** — chart candles from the DB
- **`04` backtest** — run a strategy over the DB

This notebook never writes to the database — that is `02_import_csv_to_db.ipynb`'s job.

## Flow

| # | section | what happens |
| --- | --- | --- |
| 1 | **Config** | load credentials, `EPIC`, `RESOLUTION`, the `START`/`END` window, and the `SAVE_CSV` toggle from `.env` |
| 2 | **Download** | authenticate an `IGSession`, page `GET /prices/{epic}` over `[START, END]`, print the candle count, UTC range, and remaining historical-data allowance |
| 3 | **Save** | when `SAVE_CSV` is true, write a new timestamped `data/ig_<epic-slug>_<resolution-suffix>_<timestamp>.csv` (full bid/ask/mid OHLC) |
| 4 | **Preview** | DataFrame built from the candles in memory (works even if nothing was saved): `tail(10)` / `describe()` + a close-mid chart with a high/low band |

## What a run produces

- **Config** → one block: `Loaded config from …/.env`, the masked account
  line, `EPIC=… RESOLUTION=…`, the resolved `window (UTC): <start> -> <end>`,
  and `SAVE_CSV=…`.
- **Download** → `Authenticated OK`, then `<n> <RESOLUTION> candles for
  <EPIC>`, the UTC `range` covered, and `historical-data allowance: <left> /
  <total> remaining (resets in <s>s)`.
- **Save** → `wrote …/ig_<…>.csv  (<n> rows)`, or `SAVE_CSV is False -
  skipping CSV write`.
- **Preview** → a `tail(10)` table and a `describe()` table (or `Download
  returned 0 … candles` for an empty window), then a close-mid line chart —
  or `nothing to plot` if the window held no candles.

Nothing is written to `data/` unless `IG_SAVE_CSV` is `true`.

## The parameters (all read from `.env`)

| var | meaning | notes |
| --- | --- | --- |
| `IG_START` | window start, **inclusive**, UTC | required — `"YYYY-MM-DD"` or `"YYYY-MM-DDTHH:MM:SS"` |
| `IG_END` | window end, UTC | blank → now |
| `IG_RESOLUTION` | IG candle size | `SECOND`, `MINUTE`, `MINUTE_2/3/5/10/15/30`, `HOUR`, `HOUR_2/3/4`, `DAY`, `WEEK`, `MONTH` |

`IG_RESOLUTION` also selects the CSV filename suffix (`DAY` → `daily`,
`MINUTE_10` → `10min`).

## From `.env` (this folder, git-ignored — copy `.env.sample`)

| var | meaning |
| --- | --- |
| `IG_API_KEY` | API key for the environment chosen by `IG_ACCOUNT_TYPE` |
| `IG_USERNAME` / `IG_PASSWORD` | IG login (the username, not the account number) |
| `IG_ACCOUNT_TYPE` | `demo` or `live` — **fully separate** systems, keys, and data (default `demo`) |
| `IG_EPIC` | market to pull (default `IX.D.NASDAQ.IFA.IP`, IG "US Tech 100 Cash") |
| `IG_RESOLUTION` | candle size (default `DAY`) — shared with `02`/`03`/`04` |
| `IG_START` | window start, inclusive, UTC — required |
| `IG_END` | window end, UTC — blank means now |
| `IG_SAVE_CSV` | write the CSV (`true`/`false`, default `true`) |

`IG_DAYS_BACK` / `IG_SAVE_DB` are still parsed (`cfg.days_back`, `cfg.save_db`)
but no notebook reads them.

## Gotchas

- **Stale `.env` values.** `IG_START` / `IG_END` / `IG_RESOLUTION` are plain
  `.env` settings — a forgotten update silently re-downloads the same window
  (spending historical-data allowance) or the wrong resolution. Check them
  before each run; the config cell prints what it loaded so you can verify.
- **Historical-data allowance.** Demo / retail API keys cap historical price
  *points* per rolling week. A wide window at a fine resolution can exhaust it
  (`403 error.public-api.exceeded-account-historical-data-allowance`); the
  download cell prints what's left.
- **CSV accumulation.** Every `SAVE_CSV` run writes a new file; nothing is
  cleaned up automatically.
- **Times are UTC.** `START` / `END` and the stored `snapshot_time_utc` are
  IG's `snapshotTimeUTC`, not the exchange-local `snapshotTime`.

**Prerequisite:** the `igmarket` package installed —
`uv sync --extra dev` from the repo root (see the README). That brings in
`requests` / `python-dotenv` (download) and `pandas` / `matplotlib`
(preview/chart; section 4 degrades gracefully if they're missing).

## 1. Load config

Everything here comes from `.env` — edit `IG_START` / `IG_END` / `IG_RESOLUTION`
there (copy `.env.sample` if you haven't already), then run the next cell.

- **`IG_START`** — first candle to fetch, inclusive, read as UTC. Required. A
  date (`"2024-01-01"` → `00:00:00`) or a full timestamp
  (`"2024-01-01T13:30:00"`).
- **`IG_END`** — window end, UTC. Blank means *now*
  (`datetime.now(timezone.utc)`). A date with no time (`"2024-01-01"`) is
  rolled to `23:59:59` that day so the whole day is covered; a full timestamp
  is used as-is.
- **`IG_RESOLUTION`** — IG candle size (`DAY`, `HOUR`, `MINUTE_10`, …). Drives
  both the API request and the CSV filename suffix.

The cell below does the rest:

- `Config.from_env()` (`igmarket/config.py`) loads `.env` — IG credentials,
  `EPIC`, `RESOLUTION`, `START`/`END`, `SAVE_CSV`, and the `data/` dir —
  raising `FileNotFoundError` if `.env` is missing (copy `.env.sample`) or
  `KeyError` if a credential or `IG_START` is unset;
- `START` / `END` → timezone-aware `start_dt` / `end_dt` via
  `resolve_window()` (`igmarket/timeutil.py`), which applies the bare-date
  rules above;
- `CSV_PATH = cfg.csv_path_for(RESOLUTION)` — a fresh
  `data/ig_<slug>_<suffix>_<timestamp>.csv` for this run's resolution;
- a one-line summary is printed so you can sanity-check the account, epic,
  resolution, and resolved window before any API call.

> ⚠️ **Update `.env` before each run.** `IG_START` / `IG_END` / `IG_RESOLUTION`
> are `.env` settings, not notebook variables — the config cell always reads
> whatever is currently in `.env`, but a forgotten edit there silently
> re-downloads the same window (spending historical-data allowance).
>
> - **`IG_START`** — window start, inclusive, UTC. `"YYYY-MM-DD"` or
>   `"YYYY-MM-DDTHH:MM:SS"`; a bare date is `00:00:00` that day. Required.
> - **`IG_END`** — window end, UTC. Blank = now. A bare date rolls to `23:59:59`
>   that day, so `IG_END=2026-09-04` covers all of Sept 4.
> - **`IG_RESOLUTION`** — IG candle size; also selects the CSV filename suffix.
>   One of:
>   `SECOND`, `MINUTE`, `MINUTE_2`, `MINUTE_3`, `MINUTE_5`, `MINUTE_10`,
>   `MINUTE_15`, `MINUTE_30`, `HOUR`, `HOUR_2`, `HOUR_3`, `HOUR_4`, `DAY`,
>   `WEEK`, `MONTH`.

In [6]:
from igmarket.config import Config
from igmarket.timeutil import resolve_window

cfg = Config.from_env()  # credentials, EPIC, RESOLUTION, START/END, SAVE_CSV, data dir

START = cfg.start
END = cfg.end
RESOLUTION = cfg.resolution

start_dt, end_dt = resolve_window(START, END)

CSV_PATH = cfg.csv_path_for(RESOLUTION)  # fresh timestamped path for this resolution

print(f"Loaded config from {cfg.env_path.resolve()}")
print(f"user={cfg.username!r}  account_type={cfg.account_type!r}  api_key=***{cfg.api_key[-4:]}")
print(f"EPIC={cfg.epic!r}  RESOLUTION={RESOLUTION!r}")
print(f"window (UTC): {start_dt:%Y-%m-%d %H:%M:%S}  ->  {end_dt:%Y-%m-%d %H:%M:%S}")
print(f"SAVE_CSV={cfg.save_csv}")

Loaded config from C:\repos\github.com\jupyter-notebooks\.env
user='justinjingsh-demo'  account_type='demo'  api_key=***9251
EPIC='IX.D.NASDAQ.IFA.IP'  RESOLUTION='MINUTE_10'
window (UTC): 2026-08-01 00:00:00  ->  2026-09-01 23:59:59
SAVE_CSV=True


## 2. Download

Opens an authenticated `IGSession` (`igmarket/ig_session.py` — `POST /session`
to authenticate, then `GET /prices/{epic}` (`Version: 3`) with
`resolution`/`from`/`to`/`pageSize`/`pageNumber` to page through history).

First checks the historical-data allowance with `get_allowance()` — a cheap
`max=1` request (most-recent candle only, instead of a `from`/`to` range) just
to read `metadata.allowance` before committing to the real download. Then
fetches `RESOLUTION` candles for `cfg.epic` between `START` and `END` via
`get_prices()`. Prints the candle count, the UTC time range covered, and the
allowance remaining after the download — the raw `metadata.allowance` field
names (`remainingAllowance`, `totalAllowance`, `allowanceExpiry`) live in
`constants/allowance_fields.py` (`AllowanceField`).

In [7]:
from igmarket.constants.allowance_fields import AllowanceField
from igmarket.constants.ig_candle_fields import CandleField
from igmarket.ig_session import IGSession


def _print_allowance(label, allowance):
    if not allowance:
        return
    print(
        f"  historical-data allowance ({label}): {allowance.get(AllowanceField.REMAINING_ALLOWANCE)}"
        f" / {allowance.get(AllowanceField.TOTAL_ALLOWANCE)} remaining"
        f" (resets in {allowance.get(AllowanceField.ALLOWANCE_EXPIRY)}s)"
    )


ig = IGSession(cfg.api_key, cfg.username, cfg.password, cfg.account_type)

_print_allowance("before download", ig.get_allowance(cfg.epic, RESOLUTION))

#fmt = "%Y-%m-%dT%H:%M:%S"
#raw_candles, allowance = ig.get_prices(
    #cfg.epic, RESOLUTION, start_dt.strftime(fmt), end_dt.strftime(fmt)
#)

# print(f"{len(raw_candles)} {RESOLUTION} candles for {cfg.epic}")
# if raw_candles:
#     print(f"  range (UTC): {raw_candles[0][CandleField.SNAPSHOT_TIME_UTC]}  ->  {raw_candles[-1][CandleField.SNAPSHOT_TIME_UTC]}")
# _print_allowance("after download", allowance)

Authenticated OK


RuntimeError: IG historical-data allowance exceeded: {"errorCode":"error.public-api.exceeded-account-historical-data-allowance"}

## 3. Save to CSV

Written only when `SAVE_CSV` (`IG_SAVE_CSV` in `.env`) is true — otherwise the
cell prints a skip line and the candles stay in memory for the preview below.
Load the CSV into `data/ig_market_data.db` afterwards with
`02_import_csv_to_db.ipynb`.

### CSV — full bid/ask/mid OHLC

Written only when `SAVE_CSV` is true. Row-flattening lives in
`igmarket/candle_csv.py` (`candle_to_row()`); the raw IG field-name constants
(`CandleField`, `PriceField`) are in
`igmarket/constants/ig_candle_fields.py`, and the column headers in
`igmarket/constants/csv_headers.py` (`CSV_HEADERS`) — imported by the cell
below so the header row and the values can't drift apart. Columns:

- `snapshot_time_utc` — IG's `snapshotTimeUTC`, ISO `YYYY-MM-DDTHH:MM:SS`, UTC
  (not the exchange-local `snapshotTime`)
- `{open,high,low,close}_{bid,ask,mid}_price` — 12 price columns; `mid` is
  derived (`(bid + ask) / 2`), not requested separately
- `last_traded_volume`

The target is `CSV_PATH`, built in section 1 by
`cfg.csv_path_for(RESOLUTION)`:
`data/ig_<epic-slug>_<resolution-suffix>_<timestamp>.csv`. `_epic_slug()` in
`igmarket/config.py` takes the instrument segment of the dot-separated `EPIC`
(`IX.D.NASDAQ.IFA.IP` → `nasdaq`); the suffix comes from
`igmarket/constants/csv_filename_suffix.py`'s `RESOLUTION_CSV_SUFFIX`
(`DAY` → `daily`, `MINUTE_10` → `10min`). A fixed path is never overwritten,
so successive runs' CSVs accumulate in `data/` and are never cleaned up
automatically.

In [ ]:
# --- CSV (full bid/ask/mid OHLC) ----------------------------------------------
def write_csv():
    if not cfg.save_csv:
        print("SAVE_CSV is False - skipping CSV write")
        return

    import csv

    from igmarket.candle_csv import candle_to_row
    from igmarket.constants.csv_headers import CSV_HEADERS

    with CSV_PATH.open("w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(CSV_HEADERS)
        for c in raw_candles:
            writer.writerow(candle_to_row(c))
    print(f"wrote {CSV_PATH.resolve()}  ({len(raw_candles)} rows)")


write_csv()

## 4. preview + close (mid) chart

Needs `pandas` and `matplotlib` (`%pip install pandas matplotlib`). Builds a
frame straight from the candles just downloaded (not from disk), so it works
whether or not `SAVE_CSV` wrote a file this run: a tail-10 table
+ `describe()`, then a close (mid) line chart with a shaded high/low band.

In [ ]:
try:
    import pandas as pd
except ModuleNotFoundError:
    print("pandas not installed - run:  %pip install pandas")
    df = None
else:
    from igmarket.candle_csv import candle_to_row
    from igmarket.constants.csv_headers import CSV_HEADERS

    # Straight from the candles just downloaded, so the preview and chart work
    # even when SAVE_CSV is False (nothing written to disk).
    df = pd.DataFrame((candle_to_row(c) for c in raw_candles), columns=CSV_HEADERS)
    df["snapshot_time_utc"] = pd.to_datetime(df["snapshot_time_utc"])
    df = df.set_index("snapshot_time_utc").sort_index()
    if df.empty:
        print(f"Download returned 0 {RESOLUTION} candles - nothing to preview.")
    else:
        display(df.tail(10))
        display(df.describe())

In [ ]:
if df is None:
    print("Run the pandas cell above first.")
elif df.empty:
    print(
        f"No {RESOLUTION} candles between {START} and {END or 'now'} - "
        f"nothing to plot (e.g. a weekend / market-closed window); widen START/END."
    )
else:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        print("matplotlib not installed - run:  %pip install matplotlib")
    else:
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(df.index, df["close_mid_price"], color="#26a69a")
        ax.fill_between(df.index, df["low_mid_price"], df["high_mid_price"], color="#26a69a", alpha=0.15)
        ax.set_title(f"{cfg.epic}  {RESOLUTION}  close (mid)  -  {START} to {END or 'now'}")
        ax.set_ylabel("price (mid)")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()